# **AI TECH INSTITUTE** · *Intermediate AI & Data Science*
### ML Pipelines & Model Deployment
**Instructor:** Amir Charkhi | **Focus:** From Training to Production

---

## 🎯 What You'll Learn

- Build scikit-learn pipelines
- Save and load trained models
- Deploy models as APIs
- Create simple web interfaces
- Best practices for production

---

## 💡 The Journey

```
DEVELOPMENT                          PRODUCTION
┌─────────────┐                     ┌─────────────┐
│   Notebook  │                     │  Real Users │
│             │                     │             │
│ Train model │  ─────────────────> │ Make        │
│ Experiment  │  How do we          │ Predictions │
│ Evaluate    │  get here?          │ Get results │
└─────────────┘                     └─────────────┘
```

**Answer: Pipelines + Deployment**

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
import joblib
import pickle
import json

---
## 2. The Problem: Traditional Workflow

### ❌ Without Pipelines

```python
# Training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
model = RandomForestClassifier()
model.fit(X_train_scaled, y_train)

# Prediction (DANGEROUS!)
X_new_scaled = scaler.transform(X_new)  # Must remember scaler!
prediction = model.predict(X_new_scaled)
```

**Problems:**
1. Must track scaler AND model separately
2. Easy to forget preprocessing steps
3. Order matters - can make mistakes
4. Hard to deploy

---

### ✅ With Pipelines

```python
# Training
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])
pipeline.fit(X_train, y_train)

# Prediction (SAFE!)
prediction = pipeline.predict(X_new)  # Handles everything!
```

**Benefits:**
1. Everything in one object
2. Preprocessing automatic
3. Can't forget steps
4. Easy to deploy

---

## 3. Load and Prepare Data

In [ ]:
# Load iris dataset
iris = load_iris()
X = iris.data
y = iris.target

In [ ]:
# Create DataFrame for easier handling
df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = y
df.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape, X_test.shape

---
## 4. Building a Pipeline

### 📖 Pipeline Anatomy

```
Pipeline = Sequence of steps

┌──────────────┐
│ Raw Features │
└──────┬───────┘
       │
       ↓
┌──────────────┐
│  Step 1:     │  ← Transformer (fit + transform)
│  Scaler      │
└──────┬───────┘
       │
       ↓
┌──────────────┐
│  Step 2:     │  ← Transformer (fit + transform)
│  PCA         │
└──────┬───────┘
       │
       ↓
┌──────────────┐
│  Final Step: │  ← Estimator (fit + predict)
│  Model       │
└──────────────┘
```

**Rules:**
- All steps except last: Transformers (have fit & transform)
- Last step: Estimator (has fit & predict)
- Each step has a name (string) and object (instance)

---

### 🔨 Create Pipeline

In [ ]:
# Define pipeline steps
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [ ]:
# View pipeline structure
pipeline

### 🏋️ Train Pipeline

In [ ]:
# Fit pipeline (scales then trains)
pipeline.fit(X_train, y_train)

💡 **Behind the scenes:**
1. Scaler fits on X_train
2. Scaler transforms X_train
3. Classifier trains on scaled data

### 🎯 Make Predictions

In [ ]:
# Predict (automatically scales first!)
y_pred = pipeline.predict(X_test)

In [ ]:
# Evaluate
accuracy = accuracy_score(y_test, y_pred)
f"Accuracy: {accuracy:.1%}"

### 📊 Detailed Results

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

---
## 5. Accessing Pipeline Components

In [ ]:
# Access scaler
scaler = pipeline.named_steps['scaler']
scaler.mean_

In [ ]:
# Access model
model = pipeline.named_steps['classifier']
model.feature_importances_

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': iris.feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance

---
## 6. Saving Models for Deployment

### 📦 Why Save Models?

```
Training (slow):                 Production (fast):
  Hours/days                       Milliseconds
  Expensive compute                Cheap compute
  Done once                        Done millions of times
  
Solution: Train once → Save → Load in production
```

---

### 💾 Method 1: Joblib (Recommended)

In [ ]:
# Save pipeline
joblib.dump(pipeline, 'iris_model.pkl')

In [ ]:
# Check file size
import os
size_mb = os.path.getsize('iris_model.pkl') / 1024 / 1024
f"Model size: {size_mb:.2f} MB"

### 📂 Method 2: Pickle

In [ ]:
# Save with pickle
with open('iris_model_pickle.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

💡 **Joblib vs Pickle:**
- Joblib: Better for large numpy arrays (scikit-learn models)
- Pickle: Standard Python, works with anything
- Both work fine for most cases

---

## 7. Loading and Using Saved Models

### 📥 Load Model

In [ ]:
# Load the saved pipeline
loaded_pipeline = joblib.load('iris_model.pkl')

### ✅ Verify It Works

In [ ]:
# Make predictions with loaded model
y_pred_loaded = loaded_pipeline.predict(X_test)

In [ ]:
# Check predictions match original
np.array_equal(y_pred, y_pred_loaded)

### 🌸 Predict Single Flower

In [ ]:
# New flower measurements
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])  # Looks like setosa

In [ ]:
# Predict species
prediction = loaded_pipeline.predict(new_flower)
species = iris.target_names[prediction[0]]
species

In [ ]:
# Get prediction probabilities
probabilities = loaded_pipeline.predict_proba(new_flower)[0]

pd.DataFrame({
    'Species': iris.target_names,
    'Probability': probabilities
}).sort_values('Probability', ascending=False)

---
## 8. Create Prediction Function

### 🎯 Production-Ready Function

In [ ]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
    """
    Predict iris species from measurements
    
    Returns: dict with prediction and probabilities
    """
    # Load model
    model = joblib.load('iris_model.pkl')
    
    # Prepare input
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    
    # Predict
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    # Format output
    return {
        'species': iris.target_names[prediction],
        'confidence': float(probabilities.max()),
        'probabilities': {
            species: float(prob) 
            for species, prob in zip(iris.target_names, probabilities)
        }
    }

In [ ]:
# Test function
result = predict_iris(5.1, 3.5, 1.4, 0.2)
result

In [ ]:
# Another example
result = predict_iris(6.3, 3.3, 6.0, 2.5)
result

---
## 9. Simple API with Flask

### 🌐 What's an API?

```
API = Application Programming Interface

Allows other programs to use your model:

┌──────────────┐                    ┌──────────────┐
│ Web App      │  ──HTTP Request──> │ Your API     │
│ Mobile App   │  <─HTTP Response─  │ (Flask)      │
│ Other Code   │                    │              │
└──────────────┘                    │ Loads model  │
                                    │ Makes predict│
                                    │ Returns JSON │
                                    └──────────────┘
```

---

### 📝 Create Flask API

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

# Create Flask app
app = Flask(__name__)

# Load model at startup
model = joblib.load('iris_model.pkl')
species_names = ['setosa', 'versicolor', 'virginica']

@app.route('/')
def home():
    return "Iris Prediction API is running!"

@app.route('/predict', methods=['POST'])
def predict():
    # Get data from request
    data = request.get_json()
    
    # Extract features
    features = np.array([[
        data['sepal_length'],
        data['sepal_width'],
        data['petal_length'],
        data['petal_width']
    ]])
    
    # Make prediction
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    # Return results
    return jsonify({
        'species': species_names[prediction],
        'confidence': float(probabilities.max()),
        'probabilities': {
            name: float(prob)
            for name, prob in zip(species_names, probabilities)
        }
    })

if __name__ == '__main__':
    app.run(debug=True, port=5000)

### 🚀 Run the API

```bash
# In terminal:
python app.py
```

### 📡 Test the API

```python
import requests

# Send prediction request
response = requests.post('http://localhost:5000/predict', json={
    'sepal_length': 5.1,
    'sepal_width': 3.5,
    'petal_length': 1.4,
    'petal_width': 0.2
})

print(response.json())
```

---

## 10. Simple Web Interface with Streamlit

### 🎨 What's Streamlit?

```
Streamlit = Quick way to build web apps

Input:
  Python script with widgets
  
Output:
  Interactive web application
  
Perfect for:
  ML demos, dashboards, internal tools
```

---

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import joblib
import numpy as np

# Page config
st.set_page_config(page_title="Iris Predictor", page_icon="🌸")

# Load model
@st.cache_resource
def load_model():
    return joblib.load('iris_model.pkl')

model = load_model()
species_names = ['Setosa', 'Versicolor', 'Virginica']

# Title
st.title('🌸 Iris Species Predictor')
st.write('Enter flower measurements to predict the species')

# Create two columns
col1, col2 = st.columns(2)

with col1:
    sepal_length = st.slider('Sepal Length (cm)', 4.0, 8.0, 5.1, 0.1)
    sepal_width = st.slider('Sepal Width (cm)', 2.0, 5.0, 3.5, 0.1)

with col2:
    petal_length = st.slider('Petal Length (cm)', 1.0, 7.0, 1.4, 0.1)
    petal_width = st.slider('Petal Width (cm)', 0.1, 3.0, 0.2, 0.1)

# Predict button
if st.button('Predict Species', type='primary'):
    # Prepare features
    features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    
    # Make prediction
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    # Display results
    st.success(f'Predicted Species: **{species_names[prediction]}**')
    st.write(f'Confidence: {probabilities.max():.1%}')
    
    # Show all probabilities
    st.write('### Probabilities')
    for name, prob in zip(species_names, probabilities):
        st.progress(prob, text=f'{name}: {prob:.1%}')

### 🚀 Run the Streamlit App

```bash
# In terminal:
streamlit run streamlit_app.py
```

Browser opens automatically with interactive interface!

---

## 11. Production Best Practices

### ✅ Checklist

```
BEFORE DEPLOYMENT:

□ Save complete pipeline (not just model)
□ Save preprocessing objects (scaler, encoder, etc.)
□ Save feature names/order
□ Version your models (model_v1.pkl, model_v2.pkl)
□ Test loaded model matches training performance
□ Handle missing values in production
□ Handle out-of-range values
□ Log predictions and inputs
□ Monitor model performance over time
□ Plan for model retraining
```

---

### 📦 Save Model Metadata

In [ ]:
# Create metadata file
metadata = {
    'model_version': '1.0',
    'training_date': '2024-11-27',
    'feature_names': iris.feature_names,
    'target_names': iris.target_names.tolist(),
    'accuracy': float(accuracy),
    'n_training_samples': len(X_train),
    'model_type': 'RandomForestClassifier',
    'preprocessing': ['StandardScaler']
}

In [ ]:
# Save metadata
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# Load and verify metadata
with open('model_metadata.json', 'r') as f:
    loaded_metadata = json.load(f)

loaded_metadata

### 🛡️ Input Validation

In [ ]:
def validate_and_predict(sepal_length, sepal_width, petal_length, petal_width):
    """
    Production-ready prediction with validation
    """
    # Validation ranges (from training data)
    ranges = {
        'sepal_length': (4.0, 8.0),
        'sepal_width': (2.0, 5.0),
        'petal_length': (1.0, 7.0),
        'petal_width': (0.1, 3.0)
    }
    
    # Check ranges
    inputs = {
        'sepal_length': sepal_length,
        'sepal_width': sepal_width,
        'petal_length': petal_length,
        'petal_width': petal_width
    }
    
    for name, value in inputs.items():
        min_val, max_val = ranges[name]
        if not (min_val <= value <= max_val):
            return {
                'error': f'{name} must be between {min_val} and {max_val}',
                'status': 'invalid_input'
            }
    
    # Make prediction
    try:
        result = predict_iris(sepal_length, sepal_width, petal_length, petal_width)
        result['status'] = 'success'
        return result
    except Exception as e:
        return {
            'error': str(e),
            'status': 'prediction_error'
        }

In [ ]:
# Test valid input
validate_and_predict(5.1, 3.5, 1.4, 0.2)

In [ ]:
# Test invalid input
validate_and_predict(15.0, 3.5, 1.4, 0.2)

---
## 12. Summary: From Notebook to Production

### 🔄 The Complete Journey

```
1. DEVELOPMENT
   ┌─────────────────┐
   │ Jupyter         │
   │ Notebook        │
   │                 │
   │ • Load data     │
   │ • Train model   │
   │ • Evaluate      │
   └────────┬────────┘
            │
            ↓
2. PIPELINE
   ┌─────────────────┐
   │ Create Pipeline │
   │                 │
   │ Preprocessing + │
   │ Model together  │
   └────────┬────────┘
            │
            ↓
3. SAVE
   ┌─────────────────┐
   │ Save to disk    │
   │                 │
   │ • model.pkl     │
   │ • metadata.json │
   └────────┬────────┘
            │
            ↓
4. DEPLOY
   ┌─────────────────┐
   │ Production      │
   │                 │
   │ • API (Flask)   │
   │ • Web (Stream)  │
   │ • Function      │
   └─────────────────┘
```

---

### 🔑 Key Concepts

**1. Pipelines**
- Bundle preprocessing + model
- Ensures consistency
- One object for everything

**2. Saving Models**
- Use joblib for scikit-learn
- Save metadata separately
- Version your models

**3. Loading Models**
- Load once at startup (not per prediction)
- Verify it works
- Test with known inputs

**4. Deployment Options**
- Flask API: For other programs
- Streamlit: For interactive demos
- Functions: For batch processing

**5. Production Best Practices**
- Validate inputs
- Handle errors gracefully
- Log predictions
- Monitor performance

---

### 📚 Files Created

```
iris_model.pkl          # Trained pipeline
model_metadata.json     # Model information
app.py                  # Flask API
streamlit_app.py        # Streamlit web app
```

---

### 🚀 Next Steps

To use in production:

1. **Flask API:**
   ```bash
   python app.py
   # Access at http://localhost:5000
   ```

2. **Streamlit App:**
   ```bash
   streamlit run streamlit_app.py
   # Opens in browser automatically
   ```

3. **Deploy to Cloud:**
   - Heroku (free tier)
   - AWS Lambda (serverless)
   - Google Cloud Run (containers)
   - Streamlit Cloud (free for Streamlit apps)

---

**You now know how to take models from notebook to production!** 🚀

---

**AI Tech Institute** | *Building Tomorrow's AI Engineers Today*